In [18]:
ROOT_PATH = 'C:/Users/khoan/OneDrive/Documents/stock_data_scraper'
import os
os.chdir(ROOT_PATH)

In [ ]:
%load_ext autoreload
%autoreload 2

In [21]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import pandas as pd
from datetime import datetime, timedelta
import talib
from functools import reduce

In [22]:
from utils.generic_utils import SQLModule
from utils.data_utils import StockPriceProcess
from utils.constants import CURRENCY_MAPPER

In [23]:
CODES = {
    'vietnam' : ['ACB'],
    'australia' : ['TPG', 'TNE', 'SGLLV']
}
CODES_TO_COUNTRY = {v : k for k,vv in CODES.items() for v in vv}
DAYS = 365
# FIXME: Considering using different period for each stock?
PERIOD = 14 # currently work best for 16 days

In [24]:
# Get 1 year data
end_date = datetime.today().date()
start_date = end_date - timedelta(days = DAYS)
world_engine = SQLModule.get_engine(country = 'world')
# Get all exchange rate
query = f"""
    SELECT
        *
    FROM daily_average_exchange_rate_usd_based
    WHERE
        date >= DATE '{start_date}'
        AND
        date <= DATE '{end_date}'
    ORDER BY date
"""
ex_rate = pd.read_sql_query(query, world_engine)
ex_rate.set_index('date', inplace = True)
# fill nan for each rate
for col in ex_rate.columns:
    # Backfilling the variables
    ex_rate[col] = ex_rate[col].fillna(method = 'bfill').fillna(method = 'ffill')

In [ ]:
df = None
for stock_code,country in CODES_TO_COUNTRY.items():
    engine = SQLModule.get_engine(country = country)
    stock_query = f"""
        SELECT
            date,
            close
        FROM transaction
        WHERE
            stock_code = '{stock_code}'
            AND
            date >= DATE '{start_date}'
            AND
            date <= DATE '{end_date}'
        ORDER BY date
    """
    _df = pd.read_sql_query(stock_query, engine)
    # Remove nan value
    _df.index = _df['date']
    _df = _df.drop('date', axis = 1)
    _df = StockPriceProcess.remove_invalid_data(_df, country = country)

    # convert to usd
    if country != 'united_states':
        # merge with ex_rate
        _df = _df.join(ex_rate[[CURRENCY_MAPPER[country]]])
        _df['close'] = _df['close'] / _df[CURRENCY_MAPPER[country]]

    # Relative strength index
    _df['rsi'] = talib.RSI(_df['close'].to_numpy(), timeperiod = PERIOD)

    _df = _df[['close','rsi']].iloc[PERIOD:]

    # Change column to multi-index
    _df.columns = pd.MultiIndex.from_tuples([(stock_code,col) for col in _df.columns])
    df = _df if df is None else df.join(_df)
df.iloc[-5:,:]

ACB                  TPG                   TNE             \
               close        rsi     close        rsi      close        rsi   
date                                                                         
2025-02-12  1.004700  53.380434  2.752357  50.103425  20.066384  62.497057   
2025-02-13  1.005871  54.022043  2.707967  45.713635  19.979895  60.902507   
2025-02-14  1.012186  57.426235  2.773128  52.317799  20.340482  64.920971   
2025-02-17  1.016949  59.841744  2.784152  53.351781  20.366263  65.196398   
2025-02-18  1.016581  59.560587  2.812224  55.969994  20.487241  66.524637   

               SGLLV             
               close        rsi  
date                             
2025-02-12  6.531337  49.528003  
2025-02-13  6.502891  48.576283  
2025-02-14  6.500110  48.478235  
2025-02-17  6.706120  55.625573  
2025-02-18  6.680622  54.615685

In [26]:
# Visualizing price and rsi index using dual plot

fig = make_subplots(
    rows = len(CODES_TO_COUNTRY), 
    cols = 1, 
    subplot_titles = [f'[{country}] {stock_code}' for stock_code, country in CODES_TO_COUNTRY.items()], 
    specs=[[{"secondary_y": True}] for _ in range(len(CODES_TO_COUNTRY))]
)
fig.update_annotations(font = dict(size = 20))
for num,(stock_code,country) in enumerate(CODES_TO_COUNTRY.items()):
    # Get data from that stock code
    _df = df[[(stock_code, col) for col in ['close','rsi']]].droplevel(0, axis = 1)
    # merge with ex_rate
    _df = _df.join(ex_rate[[CURRENCY_MAPPER[country]]])
    _df['close'] = _df['close'] * _df[CURRENCY_MAPPER[country]]
    # Plot the price
    fig.add_trace(
        go.Scatter(
            x = _df.index, 
            y = _df['close'],  
            marker = dict(color = 'blue'),
            showlegend = False,
            name = 'Close price'
        ),
        row = num + 1, col = 1
    )

    # Then, plot the RSI
    fig.add_trace(
        go.Scatter(
            x = _df.index, 
            y = _df['rsi'],  
            marker = dict(color = 'purple'),
            showlegend = False,
            name = 'Relative Strength Index'
        ),
        row = num + 1, col = 1, secondary_y = True
    )

    # Finally, plot the RSI lines
    fig.add_shape(
        type="line",
        x0 = _df.index[0],
        x1 = _df.index[-1],
        y0 = 70,
        y1 = 70,
        line=dict(color="green", width=2, dash="dash"),
        row = num + 1, col = 1, secondary_y = True
    )

    fig.add_shape(
        type="line",
        x0 = _df.index[0],
        x1 = _df.index[-1],
        y0 = 30,
        y1 = 30,
        line=dict(color="red", width=2, dash="dash"),
        row = num + 1, col = 1, secondary_y = True
    )
    fig.update_yaxes(
        title = 'Close price', 
        tickprefix = f'{CURRENCY_MAPPER[country]} ', 
        showgrid = True, 
        gridcolor = 'gray', 
        tickfont=dict(color='blue'), 
        secondary_y = False,
        row = num + 1, col = 1
    )
fig.update_yaxes(title = 'RSI', tickmode="sync", tickfont=dict(color='purple'), secondary_y = True)
fig.update_xaxes(title = 'Date')
fig.update_layout(
    plot_bgcolor = 'white',
    paper_bgcolor = 'white',
    height = 400 * len(CODES_TO_COUNTRY),
    font = dict(size = 20),
)
fig.show()